## Time Series Trends 

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
df = pd.read_csv("../data/processed/at_risk_orders.csv", parse_dates=date_cols)

In [ ]:
df["purchase_month"] = df['order_purchase_timestamp'].dt.to_period('M')

In [ ]:
monthly_metrics = df.groupby("purchase_month").agg(
    total_orders = ("order_id", "nunique"),
    total_delayed_orders=("order_id", lambda x: (df.loc[x.index, 'is_delayed'].groupby(df.loc[x.index, 'order_id']).max().sum()))
).reset_index()

In [ ]:
monthly_metrics["monthly_failure_rate"] = (monthly_metrics["total_delayed_orders"] / monthly_metrics["total_orders"] * 100).round(2)

In [ ]:
monthly_metrics["purchase_month"] = monthly_metrics["purchase_month"].astype("str")

In [ ]:
# Set the style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Use pd.to_datetime() directly
monthly_metrics['purchase_month_dt'] = pd.to_datetime(monthly_metrics['purchase_month'])

# Calculate rolling average (3-month window)
monthly_metrics['rolling_avg_3'] = monthly_metrics['monthly_failure_rate'].rolling(window=3, min_periods=1).mean()

# Create the line chart
fig, ax = plt.subplots(figsize=(12, 6))

# Plot the original data
ax.plot(monthly_metrics['purchase_month_dt'], 
        monthly_metrics['monthly_failure_rate'], 
        marker='o', 
        linewidth=1.5, 
        markersize=4, 
        color='steelblue', 
        alpha=0.7, 
        label='Monthly Failure Rate')

# Plot the rolling average (trendline)
ax.plot(monthly_metrics['purchase_month_dt'], 
        monthly_metrics['rolling_avg_3'], 
        linewidth=2.5, 
        color='crimson', 
        label='3-Month Rolling Average', 
        linestyle='--')

# Customize the chart
ax.set_xlabel('Purchase Month', fontsize=12, fontweight='bold')
ax.set_ylabel('Monthly Failure Rate (%)', fontsize=12, fontweight='bold')
ax.set_title('Order Delivery Failure Rate Trend Over Time', fontsize=14, fontweight='bold', pad=20)

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Add grid for better readability
ax.grid(True, alpha=0.3)

# Add legend
ax.legend(loc='upper left', fontsize=10)

# Add annotations for key observations
max_rate = monthly_metrics['monthly_failure_rate'].max()
max_date = monthly_metrics.loc[monthly_metrics['monthly_failure_rate'].idxmax(), 'purchase_month_dt']
ax.annotate(f'Peak: {max_rate}%', 
            xy=(max_date, max_rate), 
            xytext=(max_date, max_rate + 3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1),
            fontsize=9, 
            ha='center')

# Adjust layout to prevent label cutoff
plt.tight_layout()

# Display the chart
plt.show()

# Optional: Print summary statistics
print("\n=== Summary Statistics ===")
print(f"Average Monthly Failure Rate: {monthly_metrics['monthly_failure_rate'].mean():.2f}%")
print(f"Peak Monthly Failure Rate: {max_rate:.2f}% (occurred in {max_date.strftime('%Y-%m')})")
print(f"Lowest Monthly Failure Rate: {monthly_metrics['monthly_failure_rate'].min():.2f}%")
print(f"Standard Deviation: {monthly_metrics['monthly_failure_rate'].std():.2f}%")

##### Observation

Delivery failure rates are highly volatile (0%–100%, avg 9.5%, std 20.22%). The 100% spike in Sept 2016 suggests a data or logistics anomaly. The 3-month rolling average shows underlying performance stabilizes over time despite monthly swings.

